# Fine-tune TrOCR on RxHandBD (real handwritten prescription words)

**What this notebook does:** fine-tunes `microsoft/trocr-base-handwritten` on RxHandBD's training set (4,463 real, labeled handwritten medical prescription word images), then evaluates it on the held-out test set (1,115 images) so you can compare directly against the ClinicalRAG project's existing baselines:

| Model | Exact match | Avg character similarity |
|---|---|---|
| docTR (default recognition) | 0% | 4% |
| TrOCR pretrained (not fine-tuned) | 12.5% | 50.1% |
| **TrOCR fine-tuned on RxHandBD (this notebook)** | *TBD* | *TBD* |

**Why fine-tune at all?** The pretrained TrOCR checkpoint was trained on general English cursive (the IAM dataset) — it's never seen real medical drug names like "Nexcital" or "Losectil". Fine-tuning on RxHandBD teaches it the actual vocabulary and handwriting style it needs to recognize. This mirrors the approach used in the medical-OCR literature (e.g. Mask R-CNN + TrOCR fine-tuned on a custom prescription dataset, reported CER of 1.4%).

---

## Before you run this: upload the data to Google Drive

1. On your computer, find the folder `data/ocr_test/` inside the ClinicalRAG project (created earlier when the OCR testing was done locally). It should contain:
   - `Train_Set/` (4,463 `.jpg` images)
   - `Train_Label.csv`
   - `Test_Set/` (1,115 `.jpg` images)
   - `Test_Labels.csv`
2. Go to [drive.google.com](https://drive.google.com), and upload that **entire `ocr_test` folder** into `My Drive`, so the path in Drive is:
   ```
   MyDrive/ocr_test/Train_Set/
   MyDrive/ocr_test/Train_Label.csv
   MyDrive/ocr_test/Test_Set/
   MyDrive/ocr_test/Test_Labels.csv
   ```
   (If you'd rather use a different folder name/location, just edit `DATA_DIR` in the cell below to match.)
3. In Colab: **Runtime → Change runtime type → T4 GPU** (must be set *before* running any cell, or restart the runtime after changing it).
4. Run the cells in order, top to bottom.

**Time estimate:** roughly 30–60 minutes total on a free T4 GPU (5 epochs over ~4,463 images), versus an estimated many hours on a CPU-only machine — this is exactly why we moved this step to Colab.

In [ ]:
# Mount your Google Drive so we can read the uploaded dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Point this at wherever you uploaded the ocr_test folder in your Drive
DATA_DIR = '/content/drive/MyDrive/ocr_test'

# Where the fine-tuned model gets saved when training finishes —
# this is what you'll download afterward and bring back into the project.
OUTPUT_DIR = '/content/drive/MyDrive/trocr-finetuned-rxhandbd'

In [ ]:
# transformers pinned to 4.46.0 — Colab's latest default version restructured
# tokenizer loading (the new "TokenizersBackend") in a way that's incompatible
# with this older TrOCR checkpoint's tokenizer format. This version predates
# that change and is well-tested with TrOCR fine-tuning.
!pip install -q "transformers==4.46.0" torch torchvision rapidfuzz pandas pillow accelerate sentencepiece protobuf

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go back and enable the T4 GPU runtime!)')

In [ ]:
# Sanity check: confirm the data actually uploaded correctly before we go further
import os
import pandas as pd

train_labels = pd.read_csv(f'{DATA_DIR}/Train_Label.csv')
test_labels = pd.read_csv(f'{DATA_DIR}/Test_Labels.csv')

print('Train examples:', len(train_labels))
print('Test examples:', len(test_labels))
print('Train images on disk:', len(os.listdir(f'{DATA_DIR}/Train_Set')))
print('Test images on disk:', len(os.listdir(f'{DATA_DIR}/Test_Set')))
train_labels.head()

## Dataset class

Wraps the label CSV + image folder into something PyTorch can batch. `TrOCRProcessor` does two jobs here: turns each image into pixel values the vision encoder expects, and tokenizes the ground-truth text into token ids the decoder is trained to predict.

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class RxHandBDDataset(Dataset):
    def __init__(self, df, image_dir, processor, max_target_length=32):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(f"{self.image_dir}/{row['Images']}").convert('RGB')
        pixel_values = self.processor(image, return_tensors='pt').pixel_values.squeeze()

        labels = self.processor.tokenizer(
            str(row['Text']),
            padding='max_length',
            max_length=self.max_target_length,
        ).input_ids
        # Seq2SeqTrainer ignores -100 in the loss — mask out padding so the
        # model isn't penalized for not predicting pad tokens.
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]

        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Required decoder configuration for TrOCR generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 32
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

print(f'Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters')

In [ ]:
# Hold out 10% of the TRAINING set for validation during training
# (separate from the real Test_Set, which stays untouched until final evaluation)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train_labels, test_size=0.1, random_state=42)
print(f'Training on {len(train_df)} examples, validating on {len(val_df)}')

train_dataset = RxHandBDDataset(train_df, f'{DATA_DIR}/Train_Set', processor)
val_dataset = RxHandBDDataset(val_df, f'{DATA_DIR}/Train_Set', processor)

In [ ]:
# Metrics reported during training — same character-similarity measure
# used in the project's local OCR test scripts, so results are comparable.
from rapidfuzz.distance import Levenshtein
import numpy as np

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    similarities = [
        Levenshtein.normalized_similarity(p.lower(), l.lower())
        for p, l in zip(pred_str, label_str)
    ]
    exact = [p.lower().strip() == l.lower().strip() for p, l in zip(pred_str, label_str)]

    return {
        'char_similarity': float(np.mean(similarities)),
        'exact_match': float(np.mean(exact)),
    }

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# batch_size=8 + fp16 is a safe default for a free-tier T4's 16GB VRAM with
# trocr-base. If you hit a CUDA out-of-memory error, lower batch_size to 4.
training_args = Seq2SeqTrainingArguments(
    output_dir='/content/trocr-checkpoints',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=True,
    num_train_epochs=5,
    learning_rate=5e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# This is the actual fine-tuning run — expect ~30-60 minutes on a T4 for 5 epochs.
# Watch the eval_char_similarity / eval_exact_match numbers printed each epoch.
trainer.train()

## Final evaluation on the real, untouched test set

This is the number that's directly comparable to the project's existing baselines (docTR: 0%/4%, pretrained TrOCR: 12.5%/50.1%) — evaluated on all 1,115 test images, none of which the model has seen during training.

In [ ]:
test_dataset = RxHandBDDataset(test_labels, f'{DATA_DIR}/Test_Set', processor)
test_results = trainer.evaluate(test_dataset)
print(f"Fine-tuned TrOCR on full test set (n={len(test_labels)}):")
print(f"  Exact match:        {test_results['eval_exact_match']*100:.1f}%")
print(f"  Char similarity:    {test_results['eval_char_similarity']*100:.1f}%")

## Save the fine-tuned model to Drive

This is the part you bring back to the project. After this finishes, download the `trocr-finetuned-rxhandbd` folder from your Google Drive and place it at:
```
ClinicalRAG/models/trocr-finetuned-rxhandbd/
```
in the local project (create the `models/` folder if it doesn't exist). Let me know once it's downloaded and I'll wire it into `src/ocr/extractor.py` as the handwriting-recognition path.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}')